# EXPT 10 – Building a Simple LLM Chatbot using Gradio
**Name / Roll No(s):** `Jeevanantham K - 24BAD047`  |  **Date:** `12/10/2026`

**Runtime:** Colab → *Runtime → Change runtime type → T4 GPU* (CPU also works, just slower).

**Model:** `Qwen/Qwen2.5-0.5B-Instruct` (small, open, no API key needed).

## Pre-Lab Questions

1. **LLM:** A neural network trained on huge amounts of text to predict the next token, which lets it understand and generate human-like language (summarise, answer, translate, code).
2. **Transformer:** Uses self-attention so every token can weigh every other token in the context. This captures long-range meaning and trains in parallel, and it is the backbone of GPT, Llama, Qwen, etc.
3. **Prompt:** The input text/instructions given to the model. Its wording, context, examples and role/system message steer the tone, format and accuracy of the response.
4. **Chatbot vs rule-based program:** A rule-based program follows fixed if-else rules and only handles anticipated inputs. An LLM chatbot generalises from training data and handles open-ended language, but can be wrong or unpredictable.
5. **Streamlit / Gradio:** Python libraries to turn scripts/models into interactive web apps (text boxes, chat UI, buttons) with very little code, and to share or deploy them.

## PART A – Environment setup & model selection

In [12]:
!pip install -q transformers torch accelerate gradio

In [13]:
# Student details (required: code must contain roll numbers)
STUDENT_INFO = "Name: jeevanantham K | Roll No: 24BAD047"
print(STUDENT_INFO)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
model.eval()
print("Model loaded. Parameters: %.1fM" % (sum(p.numel() for p in model.parameters())/1e6))

Name: jeevanantham K | Roll No: 24BAD047
Using device: cuda


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded. Parameters: 494.0M


In [14]:
# Simple setup test with one prompt
msgs = [{"role": "user", "content": "Say hello in one sentence."}]
text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

Hello! How may I assist you today?


*Note on API keys:* this experiment uses an open model, so no key is needed.

## PART B – Prompt handling & response generation

In [15]:
SYSTEM_PROMPT = "You are a helpful, concise assistant."

def generate_response(user_message, history=None, max_new_tokens=256, temperature=0.7, top_p=0.9):
    """history: list of {"role": "user"/"assistant", "content": str}"""
    # --- handle empty input ---
    if user_message is None or not user_message.strip():
        return "⚠️ Please type a message."
    try:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        messages += (history or [])
        messages.append({"role": "user", "content": user_message.strip()})

        # prepare prompt in the model's chat format
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                pad_token_id=tokenizer.eos_token_id,
            )
        # decode only the newly generated tokens
        new_tokens = output_ids[0][inputs.input_ids.shape[1]:]
        return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    except Exception as e:           # basic runtime error handling
        return f"❌ Error: {e}"

print(STUDENT_INFO)
print(generate_response(""))          # empty-input test

Name: jeevanantham K | Roll No: 24BAD047
⚠️ Please type a message.


In [16]:
# At least five different prompts
test_prompts = [
    "What is the capital of France?",
    "Explain machine learning to a 10-year-old.",
    "Write a Python function to reverse a string.",
    "Give me 3 tips to prepare for exams.",
    "Translate 'Good morning' into Spanish.",
    "Who won the FIFA World Cup in 2030?",   # tests hallucination
]
results = []
for p in test_prompts:
    r = generate_response(p)
    results.append((p, r))
    print("🧑 PROMPT  :", p)
    print("🤖 RESPONSE:", r)
    print("-"*70)

🧑 PROMPT  : What is the capital of France?
🤖 RESPONSE: The capital of France is Paris.
----------------------------------------------------------------------
🧑 PROMPT  : Explain machine learning to a 10-year-old.
🤖 RESPONSE: Machine learning is like magic! It's when computers learn and do things on their own without being told how to do it. Imagine you have a toy robot that learns how to build blocks, or a computer program that learns how to play games. Machine learning uses these toys (or programs) to figure out how to play better at something by making changes over time.

For example, imagine you're building with blocks. The robot might start off trying to build a simple tower using only one block. But if the robot keeps playing with different shapes of blocks, it'll learn more and get better at stacking towers!

Machine learning can also help computers understand pictures or videos, or even tell stories about what they see. Just like you might use your imagination to create fun stor

## PART C – Chatbot interface (Gradio)

In [17]:
import gradio as gr

def chat_fn(message, history):
    # Convert Gradio history -> model format (supports both dict and tuple styles)
    hist = []
    for h in history:
        if isinstance(h, dict):
            hist.append({"role": h["role"], "content": h["content"] if isinstance(h["content"], str) else str(h["content"])})
        else:
            u, a = h
            hist += [{"role": "user", "content": u}, {"role": "assistant", "content": a}]
    return generate_response(message, hist)

demo = gr.ChatInterface(
    fn=chat_fn,
    title="Simple LLM Chatbot (Qwen2.5-0.5B)",
    description=STUDENT_INFO,
    examples=["What is a Transformer?", "Write a haiku about rain", "Explain recursion simply"],
)
# ChatInterface keeps conversation history and has Retry / Undo / Clear buttons built in.
demo.launch(share=True, debug=False)   # share=True gives a public *.gradio.live link (valid ~1 week)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://18b5e61594bbf03503.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


> The cell prints a **local URL** and a **public URL** (`https://xxxx.gradio.live`). Open it in a browser tab, chat, and take screenshots (Part C). Stop the cell afterwards (■ button).
>
> Test the **Clear** button and check follow-up questions work (e.g. ask “Who is Einstein?” then “Where was he born?”), which shows conversation history.

## PART D – Testing, limitations & deployment

In [18]:
import pandas as pd
tests = {
 "Factual":        ["Who wrote Romeo and Juliet?", "What is the boiling point of water in Celsius?"],
 "Conversational": ["Hi, how are you today?", "I feel stressed about exams."],
 "Task-oriented":  ["Write an email asking for leave for 2 days.", "Summarise: 'Transformers use self-attention to process tokens in parallel.'"],
}
rows = []
for cat, ps in tests.items():
    for p in ps:
        rows.append({"Category": cat, "Prompt": p, "Response": generate_response(p)})
df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 200)
df.to_csv("sample_outputs.csv", index=False)
df

,Category,Prompt,Response
0,Factual,Who wrote Romeo and Juliet?,Romeo and Juliet was written by William Shakespeare.
1,Factual,What is the boiling point of water in Celsius?,The boiling point of water is 100 degrees Celsius (212 degrees Fahrenheit).
2,Conversational,"Hi, how are you today?","I'm doing well, thank you for asking! How about you?"
3,Conversational,I feel stressed about exams.,It's normal to feel stressed before exams. Here are some tips that might help:\n\n1. Set clear goals and expectations for yourself.\n2. Create a study schedule and stick to it.\n3. Take breaks thr...
4,Task-oriented,Write an email asking for leave for 2 days.,"Subject: Request for Leave\n\nDear [Recipient's Name],\n\nI hope this message finds you well.\n\nI am writing to request a two-day absence from work due to personal circumstances. I have been feel..."
5,Task-oriented,Summarise: 'Transformers use self-attention to process tokens in parallel.',"Transformers utilize self-attention mechanisms to process tokens concurrently, enhancing their ability to handle large-scale language modeling tasks."


### Observations & limitations (edit after reading your own outputs)
| Check | Observation |
|---|---|
| Relevance / readability | Responses were mostly relevant, short and readable for factual and task prompts. |
| Incorrect output | The small 0.5B model can hallucinate (e.g. the "World Cup 2030" prompt) and may make factual or arithmetic mistakes. |
| Bias / safety | Small models may reflect training-data bias and can produce unsafe text if pushed; no strong safety filtering. |
| Latency | Slower on CPU; fast on a T4 GPU. |
| Other | Limited context length, no live knowledge, may be inconsistent between runs (sampling). |

### Files for GitHub / Hugging Face Spaces deployment (no secrets inside)

In [19]:
%%writefile app.py
import torch, gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

STUDENT_INFO = "Name: <<YOUR NAME>> | Roll No: <<YOUR ROLL NO>>"
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(device).eval()
SYSTEM_PROMPT = "You are a helpful, concise assistant."

def generate_response(msg, history):
    if not msg or not msg.strip():
        return "Please type a message."
    try:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        for h in history:
            if isinstance(h, dict):
                messages.append({"role": h["role"], "content": str(h["content"])})
            else:
                messages += [{"role": "user", "content": h[0]}, {"role": "assistant", "content": h[1]}]
        messages.append({"role": "user", "content": msg})
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=True,
                                 temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    except Exception as e:
        return f"Error: {e}"

gr.ChatInterface(generate_response, title="Simple LLM Chatbot", description=STUDENT_INFO).launch()

Overwriting app.py


In [20]:
%%writefile requirements.txt
transformers
torch
accelerate
gradio

Overwriting requirements.txt


In [21]:
%%writefile README.md
# Simple LLM Chatbot (Gradio)
Chatbot built with Qwen2.5-0.5B-Instruct + Hugging Face Transformers + Gradio.
Run: `pip install -r requirements.txt && python app.py`
No API keys are used or stored in this repository.

Overwriting README.md


**Deploy on Hugging Face Spaces (free):** huggingface.co → *New Space* → SDK **Gradio** → upload `app.py`, `requirements.txt`, `README.md` → wait for build → copy the Space URL.
**GitHub:** create a repo, upload the same 3 files + screenshots + `sample_outputs.csv`, copy the repo link. (If deployment isn't possible, the `gradio.live` link from Part C / local demo is acceptable.)

In [22]:
# Download your files from Colab
from google.colab import files
for f in ["app.py","requirements.txt","README.md","sample_outputs.csv"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Post-Lab Questions

1. **Tokenizer:** Splits text into tokens and maps them to integer IDs the model understands (and converts output IDs back to text).
2. **How an LLM responds:** The prompt is tokenised → embedded → passed through Transformer layers with self-attention → the model outputs a probability distribution for the next token → one token is chosen (greedy/sampling) → appended and repeated until an end token or the length limit.
3. **Streamlit/Gradio:** Provide a quick web UI (chat box, buttons, history) so users can interact with the model in a browser without writing HTML/JS, and make it easy to demo and deploy.
4. **API key security:** A leaked key lets others use your account, causing cost, quota abuse and data exposure. GitHub repos are often public/scraped, so use environment variables or secrets managers.
5. **Conversation history:** The earlier user/assistant turns sent along with each new prompt. It gives context, so follow-ups like "where was *he* born?" are understood and replies stay coherent.
6. **Limitations:** (i) Hallucination: confident but wrong answers. (ii) Bias/safety issues, limited context window, latency and compute cost, no real-time knowledge.
7. **Testing relevance:** Use varied prompts (factual, conversational, task), compare against known answers, check consistency across runs, have human raters score relevance/readability, or use metrics like ROUGE/BERTScore on reference answers.
8. **Local vs deployed:** Local runs only on your machine (localhost) and is accessible only to you. Deployed runs on a server (e.g. HF Spaces) with a public URL, available to anyone, always on, with its own resources and secrets management.